In [2]:
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

Matplotlib is building the font cache; this may take a moment.


In [3]:
# Titanic veri setini yükle
df = sns.load_dataset('titanic')
print("Veri başarıyla yüklendi!")

Veri başarıyla yüklendi!


In [4]:
# Verinin ilk 5 satırını göster
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [5]:
print(df.isnull().sum())

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


In [6]:
# 'age' sayısal olduğu için medyan ile dolduruyoruz
df['age'] = df['age'].fillna(df['age'].median())

# 'deck' kategorik olduğu için mod (en sık tekrarlayan değer) ile dolduruyoruz
# [0] eki, mod fonksiyonunun bir liste döndürmesi ve bizim ilk değeri almak istememizden kaynaklı
df['deck'] = df['deck'].fillna(df['deck'].mode()[0])

# 'embark_town' için de aynı şeyi yapalım
df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode()[0])

print("Boşluklar dolduruldu!")

Boşluklar dolduruldu!


In [7]:
print(df.isnull().sum())

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       2
class          0
who            0
adult_male     0
deck           0
embark_town    0
alive          0
alone          0
dtype: int64


In [8]:
# Unuttuğumuz 'embarked' sütununu da mod ile dolduralım
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Artık her şeyin 0 olduğunu teyit edelim
print(df.isnull().sum())

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
deck           0
embark_town    0
alive          0
alone          0
dtype: int64


In [9]:
# 'sex' ve 'embarked' gibi kategorik sütunları sayısal hale getirelim
# pd.get_dummies, bu sütunları her bir seçenek için ayrı bir sütuna dönüştürür (One-Hot Encoding)
df = pd.get_dummies(df, columns=['sex', 'embarked', 'who', 'class', 'deck', 'embark_town', 'alive'], drop_first=True)

# Bakalım veri ne hale geldi?
df.head()

,survived,pclass,age,sibsp,parch,fare,adult_male,alone,sex_male,embarked_Q,...,class_Third,deck_B,deck_C,deck_D,deck_E,deck_F,deck_G,embark_town_Queenstown,embark_town_Southampton,alive_yes
0,0,3,22.0,1,0,7.2500,True,False,True,False,...,True,False,True,False,False,False,False,False,True,False
1,1,1,38.0,1,0,71.2833,False,False,False,False,...,False,False,True,False,False,False,False,False,False,True
2,1,3,26.0,0,0,7.9250,False,True,False,False,...,True,False,True,False,False,False,False,False,True,True
3,1,1,35.0,1,0,53.1000,False,False,False,False,...,False,False,True,False,False,False,False,False,True,True
4,0,3,35.0,0,0,8.0500,True,True,True,False,...,True,False,True,False,False,False,False,False,True,False


In [10]:
# 'age' ve 'fare' sütunlarını standartlaştıralım (ortalama 0, standart sapma 1 olacak)
scaler = StandardScaler()
df[['age', 'fare']] = scaler.fit_transform(df[['age', 'fare']])

# İşlem tamam, verinin son haline bakalım
df.head()

,survived,pclass,age,sibsp,parch,fare,adult_male,alone,sex_male,embarked_Q,...,class_Third,deck_B,deck_C,deck_D,deck_E,deck_F,deck_G,embark_town_Queenstown,embark_town_Southampton,alive_yes
0,0,3,-0.565736,1,0,-0.502445,True,False,True,False,...,True,False,True,False,False,False,False,False,True,False
1,1,1,0.663861,1,0,0.786845,False,False,False,False,...,False,False,True,False,False,False,False,False,False,True
2,1,3,-0.258337,0,0,-0.488854,False,True,False,False,...,True,False,True,False,False,False,False,False,True,True
3,1,1,0.433312,1,0,0.420730,False,False,False,False,...,False,False,True,False,False,False,False,False,True,True
4,0,3,0.433312,0,0,-0.486337,True,True,True,False,...,True,False,True,False,False,False,False,False,True,False


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Hedef değişkenimiz 'survived' (hayatta kaldı mı?)
X = df.drop('survived', axis=1) # Diğer tüm sütunlar
y = df['survived']              # Tahmin etmeye çalıştığımız şey

# Veriyi eğitim ve test olarak ayıralım
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modeli eğit
model = LogisticRegression()
model.fit(X_train, y_train)

# Tahmin yap
y_pred = model.predict(X_test)

# Başarı oranına bak
print("Model Accuracy:", accuracy_score(y_test, y_pred))

Model Accuracy: 1.0


In [12]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Dengesiz veri üret (1000 örnekten sadece %5'i pozitif sınıf olsun)
X, y = make_classification(n_samples=1000, n_classes=2, weights=[0.95, 0.05], random_state=42)

# 2. Modeli eğit
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression()
model.fit(X_train, y_train)

# 3. Accuracy'ye bak (Yüksek çıkacak, şaşırma!)
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

# 4. Asıl meseleyi görmek için Confusion Matrix ve Rapor'a bak
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nDetaylı Rapor:\n", classification_report(y_test, y_pred))

Accuracy: 0.935

Confusion Matrix:
 [[186   3]
 [ 10   1]]

Detaylı Rapor:
               precision    recall  f1-score   support

           0       0.95      0.98      0.97       189
           1       0.25      0.09      0.13        11

    accuracy                           0.94       200
   macro avg       0.60      0.54      0.55       200
weighted avg       0.91      0.94      0.92       200

